In [ ]:
from src.web_scraper.sitemap_scraper import scrape_sitemap, process_sitemap_urls, scrape_multiple_links, save_raw_html_outputs
from src.utils.utils import generate_directories

In [ ]:
# Generate directories needed for project
# TODO: Get rid of this, generate the directories in the function itself
generate_directories()

### Scrape Sitemap & Save Locally
This will be used to embed the raw HTMLs

#### TODO: 
1. Make an orchestration function for this 
2. Save sitemap locally so I don't have to keep rerunning and scraping stuff I already have (for now)
3. Add missing links that I know we somehow missed in the site map

In [ ]:
# sitemap_url = 'https://wildfrostwiki.com/sitemap.xml'
# sitemap_urls = scrape_sitemap(sitemap_url)

In [ ]:
# urls = process_sitemap_urls(sitemap_urls)

In [ ]:
# html_outputs = await scrape_multiple_links(urls)

In [ ]:
# raw_html_subdirectory = 'raw_htmls'
# save_raw_html_outputs(html_outputs, raw_html_subdirectory)

### Scrape and save the sites in accordance to the ontology

Schemas:
1. Cards Schema: https://wildfrostwiki.com/index.php?title=Baby_Snowbo; grab the table at the end, create a folder structure based on that
2. Fights & Boss Battles: https://wildfrostwiki.com/The_Bog_Berries; grab the Map Events Table at the end
3. Charms: https://wildfrostwiki.com/Charms; another table
4. Stats, Buffs, Debuffs: https://wildfrostwiki.com/Stats
5. Keywords: https://wildfrostwiki.com/Keywords; relations between stats

In [ ]:
from src.data_processing.cards import CardType, CardInfo

In [ ]:
from src.data_processing.generate_schemas import generate_card_type_html_schema

In [ ]:
card_type_schema = generate_card_type_html_schema()

In [ ]:
import os
import json

filename = '../data/schemas/'
schema_filename = os.path.join(filename,'card_type_schema.json')
os.makedirs(filename, exist_ok=True)

with open(schema_filename,'w',encoding='utf-8') as f:
    json.dump(card_type_schema, f, indent=4)

In [ ]:
for k, v in card_type_schema.items():
    print(f'{k}: {v}')

In [ ]:
base_url = 'https://wildfrostwiki.com'

In [ ]:
import re

def clean_name_for_url(name: str) -> str:
    """Clean card name for use in URLs by replacing spaces with underscores"""
    return re.sub(r'\s+', '_', name)

In [ ]:
# Need to make sure that the sub_directory is part of the link. Might just use a tuple and use tuple unpacking into the scrape multiple links function
# TODO: 
#   1. Create a dictionary where each card link is placed in a subdirectory as a key 
#   2. Take the dictionary, scrape 

card_infos  = []
for card_type, cards in card_type_schema.items():
    if card_type == 'leaders':
        continue

    for card_name in cards:
        cleaned_name = clean_name_for_url(card_name)
        card_info = CardInfo(
            card_name=card_name,
            card_type=CardType(card_type),
            card_url=f'{base_url}/{cleaned_name}'
        )
        card_infos.append(card_info)

for c in card_infos:
    print(f'{c.card_name} {c.card_url}\n')

In [ ]:
urls = [card.card_url for card in card_infos]

In [ ]:
card_types_html_outputs = await scrape_multiple_links(urls, max_concurrent=50)

In [ ]:
for card_info, html in zip(card_infos, card_types_html_outputs):
    card_info.card_html = html
    if card_info.card_html is not None:
        card_info.save_html()
        card_info.parse_html()

In [ ]:
card_infos

In [ ]:
for c in card_infos:
    print(f'{c}\n')

In [ ]:
for c in card_infos:
    print(f'{c.to_dict()}\n')

### Leaders HTML

In [ ]:
# import requests
# from bs4 import BeautifulSoup

# leaders_url = 'https://wildfrostwiki.com/Leaders'
# response = requests.get(leaders_url)
# response.raise_for_status()

# soup = BeautifulSoup(response.text, 'html.parser')

# print(soup.prettify())

### Tribe Exclusivity Check

#### Tribe Assignment For Companions

In [ ]:
import requests
from bs4 import BeautifulSoup, Comment

leaders_url = 'https://wildfrostwiki.com/Companions'
response = requests.get(leaders_url)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'html.parser')

print(soup.prettify())

# Remove comments from HTML
comments = soup.find_all(string=lambda text: isinstance(text, Comment))
for comment in comments:
    comment.extract()

# Save cleaned HTML to file
with open('../data/companion_tribe_check.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())


In [ ]:
tables = soup.find_all('table', {'class': 'wikitable sortable'})

In [ ]:
second_table = tables[1]

In [ ]:
# Find the header row and get the headers
headers = [th.text.strip() for th in second_table.find('tr').find_all('th')]

In [ ]:
headers

In [ ]:
# Find the indices of the desired columns

tribe_lookup = {}

try:
    card_name_index = headers.index('Card Name')
    tribe_exclusive_index = headers.index('Tribe-exclusive?')
except ValueError as e:
    print(f"One of the required headers was not found: {e}")
else:
    # Iterate over each row (skipping the header row)
    for row in second_table.find_all('tr')[1:]:
        cells = row.find_all(['th', 'td'])
        
        if len(cells) > max(card_name_index, tribe_exclusive_index):
            card_name = cells[card_name_index].get_text(strip=True)
            tribe_name = cells[tribe_exclusive_index].get_text(strip=True)

            # Populate the dictionary directly
            tribe_lookup[card_name] = tribe_name

            print(f"Card Name: {card_name}, Tribe-exclusive?: {tribe_name}")

In [ ]:
tribe_lookup

In [ ]:
from src.data_processing.cards import TribeExclusivity

In [ ]:
for card_info in card_infos:
    tribe_name_str = tribe_lookup.get(card_info.card_name)

    if tribe_name_str:
        try:
            # Dynamically find the correct enum member
            matching_enum = next(t for t in TribeExclusivity if t.value == tribe_name_str)
            
            # Assign the enum member to the card's field
            card_info.tribe_exclusivity = matching_enum
            
        except StopIteration:
            # This handles cases where a tribe string exists but doesn't match an enum member.
            print(f"Warning: No matching TribeExclusivity enum found for '{tribe_name_str}' for card '{card_info.card_name}'")

In [ ]:
for c in card_infos:
    print(f'{c.card_name}: {c.tribe_exclusivity}')

#### Tribe Assignment For Items

In [ ]:
import requests
from bs4 import BeautifulSoup

leaders_url = 'https://wildfrostwiki.com/Items'
response = requests.get(leaders_url)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'html.parser')

print(soup.prettify())

# Remove comments from HTML
comments = soup.find_all(string=lambda text: isinstance(text, Comment))
for comment in comments:
    comment.extract()

# Save cleaned HTML to file
with open('../data/item_tribe_check.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())


In [ ]:
tables = soup.find_all('table', {'class': 'wikitable sortable'})

In [ ]:
only_table = tables[0]
only_table

In [ ]:
# Find the header row and get the headers
headers = [th.text.strip() for th in only_table.find('tr').find_all('th')]

In [ ]:
headers

In [ ]:
# Find the indices of the desired columns
tribe_lookup_items = {}

try:
    card_name_index = headers.index('Card Name')
    tribe_exclusive_index = headers.index('Tribe-exclusive?')
except ValueError as e:
    print(f"One of the required headers was not found: {e}")
else:
    # Iterate over each row (skipping the header row)
    for row in only_table.find_all('tr')[1:]:
        cells = row.find_all(['th', 'td'])
        
        if len(cells) > max(card_name_index, tribe_exclusive_index):
            card_name = cells[card_name_index].get_text(strip=True)
            tribe_name = cells[tribe_exclusive_index].get_text(strip=True)

            # Populate the dictionary directly
            tribe_lookup_items[card_name] = tribe_name

            print(f"Card Name: {card_name}, Tribe-exclusive?: {tribe_name}")

In [ ]:
tribe_lookup_items

In [ ]:
from src.data_processing.cards import TribeExclusivity

In [ ]:
for card_info in card_infos:
    tribe_name_str = tribe_lookup_items.get(card_info.card_name)

    if tribe_name_str:
        try:
            # Dynamically find the correct enum member
            matching_enum = next(t for t in TribeExclusivity if t.value == tribe_name_str)
            
            # Assign the enum member to the card's field
            card_info.tribe_exclusivity = matching_enum
            
        except StopIteration:
            # This handles cases where a tribe string exists but doesn't match an enum member.
            print(f"Warning: No matching TribeExclusivity enum found for '{tribe_name_str}' for card '{card_info.card_name}'")

In [ ]:
for c in card_infos:
    print(f'{c.card_name}: {c.tribe_exclusivity}')

### Test Neo4j

In [ ]:
from src.neo4j_kg.neo4j_utils import create_neo4j_data

In [ ]:
# Put all the dictionary info of card_infos into a list
# I should make this a function really
cards_dict_data = [card.to_dict() for card in card_infos]

In [ ]:
cards_dict_data

In [ ]:
create_neo4j_data(cards_dict_data)

### Chunking HTML

In [1]:
from src.data_processing.html_splitter import process_html_files

In [2]:
# Example usage of the function with a list of file paths.
# Note: The file 'data/structured_outputs/items/Azul Battle Axe.html' must exist for this to run.
sample_filepaths = ['data/structured_outputs/items/Azul Battle Axe.html', 'data/structured_outputs/items/Azul Candle.html']

# Process the files and get all the chunks.
all_document_chunks = process_html_files(sample_filepaths)

# Print a summary of all chunks from all files.
if all_document_chunks:
    print("\n--- All Chunks from All Files ---")
    for i, chunk in enumerate(all_document_chunks):
        print(f"Chunk {i+1}: '{chunk.page_content}...'")
        print(f"Metadata: {chunk.metadata}")
        print("-" * 20)
    print(f"\nTotal chunks returned: {len(all_document_chunks)}")
else:
    print("\nNo chunks were created.")

Processed 'data/structured_outputs/items/Azul Battle Axe.html': 6 chunks created.
Processed 'data/structured_outputs/items/Azul Candle.html': 4 chunks created.

--- All Chunks from All Files ---
Chunk 1: 'Azul Battle Axe...'
Metadata: {'Header 1': 'Azul Battle Axe'}
--------------------
Chunk 2: 'From Wildfrost Wiki  
Jump to navigation  
Jump to search  
Azul Battle Axe  
Attack  
3  
Other Stats  
Card Description  
Apply equal to damage dealt  
Overburn  
Card Art  
is an players may accumulate over a run. This card is exclusive to the .  
Azul Battle Axe  
item  
Shademancers Tribe...'
Metadata: {'Header 1': 'Azul Battle Axe'}
--------------------
Chunk 3: 'Contents...'
Metadata: {'Header 1': 'Azul Battle Axe', 'Header 2': 'Contents'}
--------------------
Chunk 4: '1  
Strategy  
2  
Interactions  
3  
History  
4  
Other Languages  
Strategy  
The Azul Battle Axe is a highly effective offensive card, fulfilling the dual role of inflicting and damaging the target to bring it closer